# FLX-Nano — Colab Training Runner

Thin execution wrapper for GPU training. Code lives in GitHub, compute happens here, state persists in Drive.

**Runtime → Change runtime type → GPU (A100 or L4 preferred, T4 for smoke tests only)**

In [ ]:
# Cell 1: Clone repo and install
!git clone https://github.com/Unseengap/flux-model.git
%cd flux-model
!pip install -e '.[dev]' -q
!pip install datasets transformers -q

In [ ]:
#cell 1.1: Pull latest changes and reinstall FLX
import subprocess, sys, os, importlib

os.chdir("/content/flux-model")
subprocess.run(["git", "pull", "origin", "main"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"], check=True)

# Purge all flx modules so they re-import fresh from updated source.
# Must also invalidate caches and re-import the parent packages so the
# __package__ / __spec__ chain is intact for relative imports.
for mod_name in sorted(sys.modules, reverse=True):
    if mod_name.startswith("flx"):
        del sys.modules[mod_name]
importlib.invalidate_caches()

# Re-import parent packages to re-establish __package__ chain
import flx                    # noqa: F811
import flx.training           # noqa: F811

print("✓ Updated — flx modules will re-import from new source")

In [ ]:
# Cell 2: Mount Google Drive for persistent .flx state
from google.colab import drive
drive.mount('/content/drive')

FLX_HUB = '/content/drive/MyDrive/flx_state'
import os
os.makedirs(FLX_HUB, exist_ok=True)
print(f'State hub: {FLX_HUB}')

In [ ]:
# Cell 3: Verify GPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Cell 3.1: GPU optimizations — TF32, cuDNN benchmark, DataLoader config
from flx.training.utils import configure_gpu
configure_gpu()

# DataLoader settings: pin_memory overlaps CPU→GPU transfer with compute,
# num_workers parallelizes data loading on CPU cores.
USE_PIN_MEMORY = torch.cuda.is_available()
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
PREFETCH_FACTOR = 2 if NUM_WORKERS > 0 else None
PERSISTENT_WORKERS = NUM_WORKERS > 0

print(f"GPU opts: TF32={'enabled' if torch.cuda.is_available() else 'n/a'}, "
      f"pin_memory={USE_PIN_MEMORY}, num_workers={NUM_WORKERS}")

In [ ]:
# Tokenizer + data processing utilities + shared training constants
import itertools, pickle, os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained("NousResearch/Llama-2-7b-hf")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
VOCAB_SIZE = tokenizer.vocab_size  # 32000
SEQ_LEN = 512
BATCH_SIZE = 16

CKPT_DIR = f"{FLX_HUB}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

DATA_DIR = f"{FLX_HUB}/processed_data"
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Tokenizer ready — vocab_size={VOCAB_SIZE}, batch_size={BATCH_SIZE}")

class CausalLMDataset(Dataset):
    """Raw text → (input_ids, targets) for causal LM training."""
    def __init__(self, examples):
        self.examples = examples

    @classmethod
    def from_texts(cls, texts, tokenizer, max_len=512):
        examples = []
        for text in texts:
            ids = tokenizer(text, truncation=True, max_length=max_len + 1,
                            return_tensors="pt")["input_ids"][0]
            if len(ids) < 2:
                continue
            input_ids = ids[:-1]
            targets = ids[1:]
            pad_len = max_len - len(input_ids)
            if pad_len > 0:
                input_ids = F.pad(input_ids, (0, pad_len), value=tokenizer.pad_token_id)
                targets = F.pad(targets, (0, pad_len), value=-100)
            examples.append((input_ids, targets))
        return cls(examples)

    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

def save_processed(data, path):
    with open(path, "wb") as f:
        pickle.dump(data, f)
    print(f"Saved → {path} ({os.path.getsize(path) / 1e6:.1f} MB)")

def load_processed(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    print(f"Loaded cached ← {path}")
    return data

print("Data utilities ready")

In [ ]:
# Cell 4: Create FLX-Nano model
from flx.model import FLXNano
from flx.router import ThalamicRouter
from flx.thermal import ThermalEstimator
from flx.bridges import build_bridges
from flx.memory import MemoryController, EpisodicCompressor
from flx.meta_gen import MetaDeltaGenerator
from flx.serialization import save_flx, load_flx

model = FLXNano()
router = ThalamicRouter(d_model=512, cortex_names=model.cortex_names)
model.attach_router(router)

counts = model.count_parameters()
print('Parameter counts:')
for k, v in counts.items():
    print(f'  {k}: {v:,}')

## Phase 0 & 1 — Domain-Diverse Pretraining Data

Phases 0 and 1 share the same dataset: a domain-diverse mix of language, math, code, science, and reasoning text.
Streams from HuggingFace, tokenizes, and saves to Drive. Skips processing if already cached.

In [ ]:
# Process Phase 0/1 data — domain-diverse pretraining mix
# ~1M samples × 512 tokens ≈ 512M tokens for ~145M params (3.5 tok/param)
# Chinchilla optimal is ~20 tok/param but this is feasible on Colab
NUM_SAMPLES_PER_DOMAIN = 200_000  # up to 1M total across 5 domains
CACHE_PATH = f"{DATA_DIR}/phase01_pretrain_v2.pkl"

if os.path.exists(CACHE_PATH):
    pretrain_dataset = load_processed(CACHE_PATH)
    pretrain_dataset = CausalLMDataset(pretrain_dataset)
else:
    print("Processing Phase 0/1 data from HuggingFace (this takes a while first time)...")
    all_texts = []

    # ── Language (200K) — general web text from two large sources ──
    print("  Downloading language data...")
    ds_lang1 = load_dataset("DKYoon/SlimPajama-6B", split="train", streaming=True)
    lang1_texts = [x["text"] for x in itertools.islice(ds_lang1, 100_000)]
    all_texts.extend(lang1_texts)
    print(f"    → {len(lang1_texts)} SlimPajama samples")

    ds_lang2 = load_dataset("allenai/c4", "en", split="train", streaming=True)
    lang2_texts = [x["text"] for x in itertools.islice(ds_lang2, 100_000)]
    all_texts.extend(lang2_texts)
    print(f"    → {len(lang2_texts)} C4 samples")

    # ── Math (200K) — diverse math from two sources ──
    print("  Downloading math data...")
    ds_math1 = load_dataset("open-web-math/open-web-math", split="train", streaming=True)
    math1_texts = [x["text"] for x in itertools.islice(ds_math1, 100_000)]
    all_texts.extend(math1_texts)
    print(f"    → {len(math1_texts)} open-web-math samples")

    ds_math2 = load_dataset("microsoft/orca-math-word-problems-200k", split="train", streaming=True)
    math2_texts = [
        f"Problem: {x['question']}\nSolution: {x['answer']}"
        for x in itertools.islice(ds_math2, 100_000)
    ]
    all_texts.extend(math2_texts)
    print(f"    → {len(math2_texts)} orca-math samples")

    # ── Code (200K) — multiple languages from code_search_net ──
    print("  Downloading code data...")
    for lang_name, field in [("python", "whole_func_string"), ("java", "whole_func_string"),
                              ("javascript", "whole_func_string")]:
        ds_code = load_dataset("code_search_net", lang_name, split="train", streaming=True)
        n_per_lang = 200_000 // 3
        code_texts = [x[field] for x in itertools.islice(ds_code, n_per_lang)]
        all_texts.extend(code_texts)
        print(f"    → {len(code_texts)} {lang_name} code samples")

    # ── Science (200K) — wikipedia + arxiv papers ──
    print("  Downloading science data...")
    ds_wiki = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
    wiki_texts = [x["text"] for x in itertools.islice(ds_wiki, 100_000)]
    all_texts.extend(wiki_texts)
    print(f"    → {len(wiki_texts)} wikipedia samples")

    ds_arxiv = load_dataset("ccdv/arxiv-summarization", split="train", streaming=True)
    arxiv_texts = [x["article"][:2000] for x in itertools.islice(ds_arxiv, 100_000)]
    all_texts.extend(arxiv_texts)
    print(f"    → {len(arxiv_texts)} arxiv paper samples")

    # ── Reasoning (200K) — instruction-following + chain-of-thought ──
    print("  Downloading reasoning data...")
    ds_orca = load_dataset("Open-Orca/OpenOrca", split="train", streaming=True)
    orca_texts = [
        f"{x['system_prompt']}\n{x['question']}\n{x['response']}"
        for x in itertools.islice(ds_orca, 185_000)
    ]
    all_texts.extend(orca_texts)
    print(f"    → {len(orca_texts)} OpenOrca reasoning samples")

    ds_dolly = load_dataset("databricks/databricks-dolly-15k", split="train", streaming=True)
    dolly_texts = [
        f"{x['instruction']}\n{x['context']}\n{x['response']}"
        for x in itertools.islice(ds_dolly, 15_000)
    ]
    all_texts.extend(dolly_texts)
    print(f"    → {len(dolly_texts)} dolly reasoning samples")

    import random
    random.shuffle(all_texts)
    print(f"Total: {len(all_texts)} samples. Tokenizing...")

    pretrain_dataset = CausalLMDataset.from_texts(all_texts, tokenizer, max_len=SEQ_LEN)
    save_processed(pretrain_dataset.examples, CACHE_PATH)

# Train/val split (90/10) — val set is used for early stopping
from flx.training.utils import make_train_val_split
pretrain_train, pretrain_val = make_train_val_split(pretrain_dataset, val_fraction=0.1)

pretrain_loader = DataLoader(
    pretrain_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
)
pretrain_val_loader = DataLoader(
    pretrain_val, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
)
print(f"Phase 0/1 DataLoader ready: {len(pretrain_train)} train, {len(pretrain_val)} val, "
      f"{len(pretrain_loader)} batches ({len(pretrain_dataset)} total × 512 tok ≈ "
      f"{len(pretrain_dataset) * 512 / 1e6:.0f}M tokens)")

In [ ]:
# Phase 0 training — Cortex Specialization (COMPLETED)
# Phase 0 finished at step 25k with pred≈3.7, saved to nano_phase0.flx
# To re-run from scratch: remove resume_from_checkpoint, set max_steps=25_000
from flx.training.phase0_cortex import train_phase0

history = train_phase0(
    model, pretrain_loader,
    val_dataloader=pretrain_val_loader,
    num_epochs=10, lr=1e-4,
    patience=3,
    checkpoint_dir=CKPT_DIR,
    checkpoint_every=5_000,
    max_steps=25_000,
    warmup_steps=500,
    device=DEVICE, log_every=100,
)
save_flx(model, f'{FLX_HUB}/nano_phase0.flx')
print(f"Phase 0 done — {len(history)} steps, final loss: {history[-1]['total_loss']:.4f}")

## Phase 1 — Delta-Receptive Pretraining

Uses the same domain-diverse data as Phase 0. Router is frozen; cortex bases learn to accept delta modifications.

In [ ]:
# Phase 1 training — Delta-Receptive Pretraining
from flx.training.phase1_delta import train_phase1

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase0.flx', device=DEVICE)
history = train_phase1(
    model, pretrain_loader,
    val_dataloader=pretrain_val_loader,
    num_epochs=10, lr=5e-5,
    patience=3,
    checkpoint_dir=CKPT_DIR,
    checkpoint_every=5_000,
    max_steps=30_000,
    warmup_steps=500,
    device=DEVICE, log_every=100,
)
save_flx(model, f'{FLX_HUB}/nano_phase1.flx')
print(f"Phase 1 done — {len(history)} steps, final loss: {history[-1]['total_loss']:.4f}")

## Phase 2 — Thermal Routing + Bridges

Needs **difficulty-stratified** data so τ learns when to think harder vs. coast.
Mix of easy (simple text), medium (grade-school math, basic code), and hard (competition math/code).

In [ ]:
# Process Phase 2 data — difficulty-stratified mix (~250K samples)
# Balanced easy/medium/hard so τ learns when to think harder vs. coast
from flx.training.utils import make_train_val_split

CACHE_PATH_P2 = f"{DATA_DIR}/phase2_thermal_v2.pkl"

if os.path.exists(CACHE_PATH_P2):
    thermal_dataset = load_processed(CACHE_PATH_P2)
    thermal_dataset = CausalLMDataset(thermal_dataset)
else:
    print("Processing Phase 2 data (difficulty-stratified)...")
    all_texts = []

    # ── EASY (~80K) — simple web text (low τ targets) ──
    print("  Easy: simple web text...")
    ds_easy1 = load_dataset("DKYoon/SlimPajama-6B", split="train", streaming=True)
    easy1_texts = [x["text"] for x in itertools.islice(ds_easy1, 40_000)]
    all_texts.extend(easy1_texts)
    print(f"    → {len(easy1_texts)} SlimPajama easy samples")

    ds_easy2 = load_dataset("allenai/c4", "en", split="train", streaming=True)
    easy2_texts = [x["text"] for x in itertools.islice(ds_easy2, 40_000)]
    all_texts.extend(easy2_texts)
    print(f"    → {len(easy2_texts)} C4 easy samples")

    # ── MEDIUM (~80K) — grade school math + basic code ──
    print("  Medium: GSM8K + MBPP + orca-math...")
    ds_gsm = load_dataset("openai/gsm8k", "main", split="train")
    gsm_texts = [f"Problem: {x['question']}\nSolution: {x['answer']}" for x in ds_gsm]
    all_texts.extend(gsm_texts)
    print(f"    → {len(gsm_texts)} GSM8K samples")

    ds_mbpp = load_dataset("google-research-datasets/mbpp", "sanitized", split="train")
    mbpp_texts = [f"# {x['prompt']}\n{x['code']}" for x in ds_mbpp]
    all_texts.extend(mbpp_texts)
    print(f"    → {len(mbpp_texts)} MBPP samples")

    ds_orca_math = load_dataset("microsoft/orca-math-word-problems-200k", split="train", streaming=True)
    orca_math_medium = [
        f"Problem: {x['question']}\nSolution: {x['answer']}"
        for x in itertools.islice(ds_orca_math, 70_000)
    ]
    all_texts.extend(orca_math_medium)
    print(f"    → {len(orca_math_medium)} orca-math medium samples")

    # ── HARD (~80K) — hard multi-domain questions + hard code (high τ) ──
    print("  Hard: MMLU auxiliary + code contests + OpenOrca hard...")
    ds_hard_q = load_dataset("cais/mmlu", "all", split="auxiliary_train", streaming=True)
    hard_q_texts = [
        f"Question: {x['question']}\nChoices: {', '.join(x['choices'])}\nAnswer: {x['choices'][x['answer']]}"
        for x in itertools.islice(ds_hard_q, 50_000)
    ]
    all_texts.extend(hard_q_texts)
    print(f"    → {len(hard_q_texts)} MMLU hard samples")

    ds_hard_code = load_dataset("deepmind/code_contests", split="train")
    hard_code_texts = [
        f"# {x['description'][:500]}\n{x['solutions']['solution'][0] if x['solutions']['solution'] else ''}"
        for x in ds_hard_code if x['solutions']['solution']
    ][:10_000]
    all_texts.extend(hard_code_texts)
    print(f"    → {len(hard_code_texts)} code contest samples")

    ds_hard_orca = load_dataset("Open-Orca/OpenOrca", split="train", streaming=True)
    hard_orca_texts = [
        f"{x['system_prompt']}\n{x['question']}\n{x['response']}"
        for x in itertools.islice(ds_hard_orca, 20_000)
    ]
    all_texts.extend(hard_orca_texts)
    print(f"    → {len(hard_orca_texts)} OpenOrca hard samples")

    import random
    random.shuffle(all_texts)
    print(f"Total: {len(all_texts)} samples. Tokenizing...")

    thermal_dataset = CausalLMDataset.from_texts(all_texts, tokenizer, max_len=SEQ_LEN)
    save_processed(thermal_dataset.examples, CACHE_PATH_P2)

# Train/val split
thermal_train, thermal_val = make_train_val_split(thermal_dataset, val_fraction=0.1)
thermal_loader = DataLoader(
    thermal_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
)
thermal_val_loader = DataLoader(
    thermal_val, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
)
print(f"Phase 2 DataLoader ready: {len(thermal_train)} train, {len(thermal_val)} val, {len(thermal_loader)} batches")

In [ ]:
# Phase 2 training — Thermal Routing + Bridges
from flx.training.phase2_thermal import train_phase2

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase1.flx', device=DEVICE)
thermal = ThermalEstimator(d_model=512)
model.attach_thermal(thermal)
bridges = build_bridges(model.cortex_names, d_model=512)
model.attach_bridges(bridges)

history = train_phase2(
    model, thermal_loader,
    val_dataloader=thermal_val_loader,
    num_epochs=5, lr=3e-5,
    patience=3,
    checkpoint_dir=CKPT_DIR,
    checkpoint_every=5_000,
    max_steps=25_000,
    warmup_steps=500,
    device=DEVICE, log_every=100,
)
save_flx(model, f'{FLX_HUB}/nano_phase2.flx')
print(f"Phase 2 done — {len(history)} steps, final loss: {history[-1]['total_loss']:.4f}")

## Phase 3 — Memory System

Needs **multi-turn conversation chains** so the memory controller learns to compress and retrieve across turns.
Converts HF conversation format → `list[list[tuple[Tensor, Tensor]]]`.

In [ ]:
# Process Phase 3 data — multi-turn conversations (20K convos, ~60K+ turns)
NUM_CONVERSATIONS = 20_000
CACHE_PATH_P3 = f"{DATA_DIR}/phase3_conversations_v2.pkl"

if os.path.exists(CACHE_PATH_P3):
    conversation_data = load_processed(CACHE_PATH_P3)
else:
    print("Processing Phase 3 conversation data...")

    # UltraChat — large multi-turn dialogue dataset (200K available)
    ds_conv = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)

    conversation_data = []
    skipped = 0
    for conv in itertools.islice(ds_conv, NUM_CONVERSATIONS * 2):  # oversample, some get skipped
        messages = conv.get("messages", [])
        if len(messages) < 2:
            skipped += 1
            continue

        turns = []
        for msg in messages:
            text = msg["content"]
            ids = tokenizer(text, truncation=True, max_length=SEQ_LEN + 1,
                            return_tensors="pt")["input_ids"]
            if ids.shape[1] < 2:
                continue
            input_ids = ids[:, :-1]  # [1, seq]
            targets = ids[:, 1:]     # [1, seq]
            # Pad to uniform length
            pad_len = SEQ_LEN - input_ids.shape[1]
            if pad_len > 0:
                input_ids = F.pad(input_ids, (0, pad_len), value=tokenizer.pad_token_id)
                targets = F.pad(targets, (0, pad_len), value=-100)
            turns.append((input_ids, targets))

        if len(turns) >= 2:
            conversation_data.append(turns)

        if len(conversation_data) >= NUM_CONVERSATIONS:
            break

    print(f"Processed {len(conversation_data)} conversations ({skipped} skipped)")
    save_processed(conversation_data, CACHE_PATH_P3)

total_turns = sum(len(c) for c in conversation_data)
print(f"Phase 3 data ready: {len(conversation_data)} conversations, "
      f"avg {total_turns / len(conversation_data):.1f} turns each, "
      f"~{total_turns * 512 / 1e6:.0f}M tokens")

In [ ]:
# Phase 3 training — Memory System
from flx.training.phase3_memory import train_phase3

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase2.flx', device=DEVICE)
mem_ctrl = MemoryController(d_model=512)
model.attach_memory(mem_ctrl)
compressor = EpisodicCompressor(d_model=512)

# Build a small val loader from Phase 2 cached data for early stopping
from flx.training.utils import make_train_val_split
_p2_cache = f"{DATA_DIR}/phase2_thermal_v2.pkl"
if os.path.exists(_p2_cache):
    _val_dataset = CausalLMDataset(load_processed(_p2_cache))
    _, _val_subset = make_train_val_split(_val_dataset, val_fraction=0.1)
    phase3_val_loader = DataLoader(
        _val_subset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
        pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
        prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
    )
    print(f"Phase 3 val loader: {len(_val_subset)} samples from Phase 2 cache")
else:
    phase3_val_loader = None
    print("No Phase 2 cache found — using train loss for early stopping")

history = train_phase3(
    model, compressor, conversation_data,
    val_dataloader=phase3_val_loader,
    num_epochs=5, lr=2e-5,
    patience=3,
    checkpoint_dir=CKPT_DIR,
    checkpoint_every=5_000,
    max_steps=20_000,
    warmup_steps=500,
    device=DEVICE, log_every=100,
)
save_flx(model, f'{FLX_HUB}/nano_phase3.flx')
print(f"Phase 3 done — {len(history)} steps, final loss: {history[-1]['pred_loss']:.4f}")

## Phase 4 — Meta-Delta Generation

Needs **challenging held-out data** the model currently struggles with.
The meta-generator learns to produce corrective deltas from accumulated prediction errors.

In [ ]:
# Process Phase 4 data — challenging evaluation data
from flx.training.utils import make_train_val_split

CACHE_PATH_P4 = f"{DATA_DIR}/phase4_challenge.pkl"

if os.path.exists(CACHE_PATH_P4):
    challenge_dataset = load_processed(CACHE_PATH_P4)
    challenge_dataset = CausalLMDataset(challenge_dataset)
else:
    print("Processing Phase 4 challenge data...")
    all_texts = []

    # MMLU — broad multi-domain knowledge
    print("  MMLU...")
    ds_mmlu = load_dataset("cais/mmlu", "all", split="test")
    mmlu_texts = [
        f"Question: {x['question']}\nChoices: {', '.join(x['choices'])}\nAnswer: {x['choices'][x['answer']]}"
        for x in ds_mmlu
    ]
    all_texts.extend(mmlu_texts)
    print(f"    → {len(mmlu_texts)} MMLU samples")

    # ARC Challenge — hard science reasoning
    print("  ARC Challenge...")
    ds_arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
    arc_texts = [
        f"Question: {x['question']}\nChoices: {', '.join(x['choices']['text'])}\nAnswer: {x['answerKey']}"
        for x in ds_arc
    ]
    all_texts.extend(arc_texts)
    print(f"    → {len(arc_texts)} ARC samples")

    # GSM8K test — held-out math reasoning
    print("  GSM8K test...")
    ds_gsm_test = load_dataset("openai/gsm8k", "main", split="test")
    gsm_test_texts = [f"Problem: {x['question']}\nSolution: {x['answer']}" for x in ds_gsm_test]
    all_texts.extend(gsm_test_texts)
    print(f"    → {len(gsm_test_texts)} GSM8K test samples")

    # ARC Easy test — broader science reasoning
    print("  ARC Easy test...")
    ds_arc_easy = load_dataset("allenai/ai2_arc", "ARC-Easy", split="test")
    arc_easy_texts = [
        f"Question: {x['question']}\nChoices: {', '.join(x['choices']['text'])}\nAnswer: {x['answerKey']}"
        for x in ds_arc_easy
    ]
    all_texts.extend(arc_easy_texts)
    print(f"    → {len(arc_easy_texts)} ARC Easy test samples")

    import random
    random.shuffle(all_texts)
    print(f"Total: {len(all_texts)} challenge samples. Tokenizing...")

    challenge_dataset = CausalLMDataset.from_texts(all_texts, tokenizer, max_len=SEQ_LEN)
    save_processed(challenge_dataset.examples, CACHE_PATH_P4)

# Train/val split
challenge_train, challenge_val = make_train_val_split(challenge_dataset, val_fraction=0.1)
challenge_loader = DataLoader(
    challenge_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
)
challenge_val_loader = DataLoader(
    challenge_val, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
)
print(f"Phase 4 DataLoader ready: {len(challenge_train)} train, {len(challenge_val)} val, {len(challenge_loader)} batches")

In [ ]:
# Phase 4 training — Meta-Delta Generation
from flx.training.phase4_meta import train_phase4

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase3.flx', device=DEVICE)
meta_gen = MetaDeltaGenerator(d_model=512, delta_rank=32, num_cortices=5)
model.attach_meta_generator(meta_gen)

history = train_phase4(
    model, meta_gen, challenge_loader,
    val_dataloader=challenge_val_loader,
    num_epochs=3, lr=1e-4,
    patience=3,
    checkpoint_dir=CKPT_DIR,
    checkpoint_every=5_000,
    max_steps=15_000,
    warmup_steps=300,
    device=DEVICE, log_every=10,
)
save_flx(model, f'{FLX_HUB}/nano_phase4.flx')
accepted = sum(1 for h in history if h.get('accepted', 0) > 0)
print(f"Phase 4 done — {len(history)} steps, {accepted}/{len(history)} deltas accepted")

## Phase 5 — Abstract Rule Induction (Few-Shot Learning)

Needs **few-shot transformation tasks**: demo pairs + test pair.
Two data sources, processed and saved to Drive:
- **ARC-AGI-1** (canonical) — 800 grid-based rule induction tasks from `lordspline/arc-agi`
- **ARC-Mega** (augmented) — streamed subset of ~5K tasks from `mindware/arc-agi-mega`

> `bicycleman15/arc_1d_8` was evaluated but uses conversation/chat format — not structured grids — so it's excluded.

In [ ]:
# Phase 5 data — Download, process, and save ARC datasets to Drive
# Sources: ARC-AGI-1 (canonical 2D grids), ARC-Mega (augmented subset)
# NOTE: 1D-ARC (bicycleman15/arc_1d_8) is in conversation/chat format, not
# structured grids — skipped in favour of more ARC-Mega tasks.
import json, random, pickle, os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

CACHE_PATH_P5 = f"{DATA_DIR}/phase5_arc.pkl"
PAD_TOKEN = 0
SEP_TOKEN = 11  # row separator for 2D grids
MAX_SEQ = SEQ_LEN  # reuse notebook SEQ_LEN (512)

def flatten_grid_2d(grid: list[list[int]]) -> list[int]:
    """Flatten a 2D ARC grid into a 1D token sequence.
    Each row separated by SEP_TOKEN. Values offset by +1 so grid-0 != pad-0."""
    tokens = []
    for i, row in enumerate(grid):
        if i > 0:
            tokens.append(SEP_TOKEN)
        tokens.extend(v + 1 for v in row)  # +1 so grid-0 != pad-0
    return tokens

def arc_task_to_tensors(demos, test_pair, max_seq=MAX_SEQ):
    """Convert one ARC task to (demo_inputs, demo_targets, test_input, test_target).

    Each demo/test grid is flattened and padded/truncated to max_seq.
    """
    demo_inputs, demo_targets = [], []
    for pair in demos:
        inp_tokens = flatten_grid_2d(pair["input"])[:max_seq]
        out_tokens = flatten_grid_2d(pair["output"])[:max_seq]
        inp_t = F.pad(torch.tensor(inp_tokens, dtype=torch.long), (0, max_seq - len(inp_tokens)), value=PAD_TOKEN)
        out_t = F.pad(torch.tensor(out_tokens, dtype=torch.long), (0, max_seq - len(out_tokens)), value=-100)
        demo_inputs.append(inp_t)
        demo_targets.append(out_t)

    test_inp = flatten_grid_2d(test_pair["input"])[:max_seq]
    test_out = flatten_grid_2d(test_pair["output"])[:max_seq]
    test_inp_t = F.pad(torch.tensor(test_inp, dtype=torch.long), (0, max_seq - len(test_inp)), value=PAD_TOKEN)
    test_out_t = F.pad(torch.tensor(test_out, dtype=torch.long), (0, max_seq - len(test_out)), value=-100)

    return demo_inputs, demo_targets, test_inp_t, test_out_t


class Phase5Dataset(Dataset):
    """Dataset for Phase 5: each item is (demo_inputs, demo_targets, test_input, test_target)."""
    def __init__(self, tasks):
        self.tasks = tasks

    def __len__(self):
        return len(self.tasks)

    def __getitem__(self, idx):
        return self.tasks[idx]


def phase5_collate(batch):
    """Custom collate: stack demo lists and test tensors across batch."""
    demo_inputs_list, demo_targets_list, test_inputs, test_targets = zip(*batch)
    max_demos = max(len(d) for d in demo_inputs_list)
    seq_len = test_inputs[0].shape[0]

    batched_demo_inputs = []
    batched_demo_targets = []
    for demo_idx in range(max_demos):
        inp_stack, tgt_stack = [], []
        for b in range(len(batch)):
            if demo_idx < len(demo_inputs_list[b]):
                inp_stack.append(demo_inputs_list[b][demo_idx])
                tgt_stack.append(demo_targets_list[b][demo_idx])
            else:
                inp_stack.append(torch.zeros(seq_len, dtype=torch.long))
                tgt_stack.append(torch.full((seq_len,), -100, dtype=torch.long))
        batched_demo_inputs.append(torch.stack(inp_stack))
        batched_demo_targets.append(torch.stack(tgt_stack))

    test_input = torch.stack(test_inputs)
    test_target = torch.stack(test_targets)
    return batched_demo_inputs, batched_demo_targets, test_input, test_target


if os.path.exists(CACHE_PATH_P5):
    all_tasks = load_processed(CACHE_PATH_P5)
    print(f"Phase 5 | Loaded {len(all_tasks)} cached tasks from Drive")
else:
    all_tasks = []

    # --- ARC-AGI-1 (canonical, ~800 tasks) ---
    # Schema: each row has columns "train" and "test" already parsed as Python lists.
    #   row["train"] = [{"input": [[int]], "output": [[int]]}, ...]
    #   row["test"]  = [{"input": [[int]], "output": [[int]]}, ...]
    # Splits: "training" (400 tasks) and "evaluation" (400 tasks).
    print("Phase 5 | Loading ARC-AGI-1 (lordspline/arc-agi)...")
    for split_name in ["training", "evaluation"]:
        count_arc = 0
        try:
            ds_arc = load_dataset("lordspline/arc-agi", split=split_name)
            for row in ds_arc:
                try:
                    train_demos = row["train"]  # already a list of dicts
                    test_pairs = row["test"]     # already a list of dicts
                    for test_pair in test_pairs:
                        result = arc_task_to_tensors(train_demos, test_pair)
                        all_tasks.append(result)
                        count_arc += 1
                except (KeyError, TypeError, IndexError) as e:
                    continue
            print(f"    -> {count_arc} ARC-AGI-1 tasks from '{split_name}'")
        except Exception as e:
            print(f"    -> ARC-AGI-1 '{split_name}' failed ({e}), skipping")

    # --- ARC-Mega (augmented subset --- stream 5K tasks) ---
    # Schema: JSONL files, each row has "train", "test", "metadata" keys.
    #   Same structure as ARC-AGI-1 --- already parsed by load_dataset.
    #   data_dir="data/train/arc_json" selects the canonical ARC JSON partition.
    ARC_MEGA_LIMIT = 5_000
    print(f"Phase 5 | Streaming ARC-Mega subset ({ARC_MEGA_LIMIT} tasks)...")
    try:
        ds_mega = load_dataset(
            "mindware/arc-agi-mega",
            data_dir="data/train/arc_json",
            split="train",
            streaming=True,
        )
        count_mega = 0
        for row in ds_mega:
            if count_mega >= ARC_MEGA_LIMIT:
                break
            try:
                # Skip multimodal tasks
                meta = row.get("metadata", {})
                if isinstance(meta, dict) and meta.get("is_multimodal", False):
                    continue
                train_demos = row["train"]
                test_pairs = row["test"]
                if not train_demos or not test_pairs:
                    continue
                for test_pair in test_pairs:
                    result = arc_task_to_tensors(train_demos, test_pair)
                    all_tasks.append(result)
                    count_mega += 1
                    if count_mega >= ARC_MEGA_LIMIT:
                        break
            except (KeyError, TypeError, IndexError):
                continue
        print(f"    -> {count_mega} ARC-Mega tasks streamed")
    except Exception as e:
        print(f"    -> ARC-Mega failed ({e}), skipping")

    random.shuffle(all_tasks)
    print(f"Phase 5 | Total: {len(all_tasks)} tasks. Saving to Drive...")
    save_processed(all_tasks, CACHE_PATH_P5)

print(f"Phase 5 | {len(all_tasks)} tasks ready")

In [ ]:
# Phase 5 DataLoaders — split and prepare for training
# Self-contained: loads cached data from Drive if the download cell was skipped
import pickle
from torch.utils.data import Dataset, DataLoader
from flx.training.utils import make_train_val_split

class Phase5Dataset(Dataset):
    def __init__(self, tasks): self.tasks = tasks
    def __len__(self): return len(self.tasks)
    def __getitem__(self, idx): return self.tasks[idx]

def phase5_collate(batch):
    demo_inputs_list, demo_targets_list, test_inputs, test_targets = zip(*batch)
    max_demos = max(len(d) for d in demo_inputs_list)
    seq_len = test_inputs[0].shape[0]
    batched_demo_inputs, batched_demo_targets = [], []
    for demo_idx in range(max_demos):
        inp_stack, tgt_stack = [], []
        for b in range(len(batch)):
            if demo_idx < len(demo_inputs_list[b]):
                inp_stack.append(demo_inputs_list[b][demo_idx])
                tgt_stack.append(demo_targets_list[b][demo_idx])
            else:
                inp_stack.append(torch.zeros(seq_len, dtype=torch.long))
                tgt_stack.append(torch.full((seq_len,), -100, dtype=torch.long))
        batched_demo_inputs.append(torch.stack(inp_stack))
        batched_demo_targets.append(torch.stack(tgt_stack))
    return batched_demo_inputs, batched_demo_targets, torch.stack(test_inputs), torch.stack(test_targets)

# Load cached tasks from Drive if not already in memory
if 'all_tasks' not in dir() or all_tasks is None:
    _cache = f"{DATA_DIR}/phase5_arc.pkl"
    with open(_cache, "rb") as f:
        all_tasks = pickle.load(f)
    print(f"Phase 5 | Loaded {len(all_tasks)} cached tasks from Drive")

phase5_dataset = Phase5Dataset(all_tasks)
phase5_train, phase5_val = make_train_val_split(phase5_dataset, val_fraction=0.1)

phase5_loader = DataLoader(
    phase5_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    collate_fn=phase5_collate,
    pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
)
phase5_val_loader = DataLoader(
    phase5_val, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
    collate_fn=phase5_collate,
    pin_memory=USE_PIN_MEMORY, num_workers=NUM_WORKERS,
    prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS,
)

# Quick sanity check
demo_in, demo_tgt, test_in, test_tgt = next(iter(phase5_loader))
print(f"Phase 5 DataLoader ready: {len(phase5_train)} train, {len(phase5_val)} val, {len(phase5_loader)} batches")
print(f"  Batch shape: {len(demo_in)} demos × [{demo_in[0].shape}], test=[{test_in.shape}]")

In [ ]:
# Phase 5 training — Abstract Rule Induction (Attempt 4: supervised consistency target)
from flx.training.phase5_abstraction import train_phase5
from flx.hypothesis import HypothesisHead

model, _, _ = load_flx(f'{FLX_HUB}/nano_phase4.flx', device=DEVICE)
hypothesis_head = HypothesisHead(d_model=512, hypothesis_dim=512)
model.attach_hypothesis_head(hypothesis_head)

history = train_phase5(
    model, hypothesis_head, phase5_loader,
    val_dataloader=phase5_val_loader,
    meta_gen=model.meta_generator,
    num_epochs=5, lr=3e-4, finetune_lr=1e-5,
    tau=0.8,
    max_loops=3,
    min_loops=1,              # force ≥1 refinement pass
    consistency_threshold=0.85,
    lambda_cons=0.5,          # supervised MSE target: cons tracks exp(-pred/2)
    lambda_loop=0.01,
    patience=5,
    checkpoint_dir=CKPT_DIR,
    device=DEVICE, log_every=10,
)
save_flx(model, f'{FLX_HUB}/nano_phase5.flx')

avg_cons = sum(h['consistency'] for h in history[-100:]) / min(len(history), 100)
avg_tgt = sum(h['cons_target'] for h in history[-100:]) / min(len(history), 100)
avg_loops = sum(h['num_loops'] for h in history[-100:]) / min(len(history), 100)
print(f"Phase 5 done — {len(history)} steps")
print(f"  Final 100-step avg: consistency={avg_cons:.3f}, target={avg_tgt:.3f}, loops={avg_loops:.1f}")
print(f"  Final pred_loss: {history[-1]['pred_loss']:.4f}")

## Profiling (Experiment 2 — Thermal Efficiency)

In [ ]:
# Cell 10: FLOP profiling
from torch.profiler import profile, record_function, ProfilerActivity

# model, _, _ = load_flx(f'{FLX_HUB}/nano_phase2.flx', device=DEVICE)
# model.eval()
# input_ids = torch.randint(0, 32000, (1, 128), device=DEVICE)
#
# with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
#              record_shapes=True, with_flops=True) as prof:
#     with record_function('flx_forward'):
#         output = model(input_ids)
#
# print(prof.key_averages().table(sort_by='flops', row_limit=20))
print('Profiling: load a trained model and uncomment above')